[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paabL/rl-hvac-lab/blob/main/fr/prototype_lab_rl.ipynb)

# Un seul thermostat peut-il chauffer mille maisons qu'il n'a jamais vues ?

**IFT3700/IFT6758 · Science des données · Labo RL**

Au cours 2, le RL servait à aligner les LLM par préférences. Aujourd'hui, on lui demande un **contrôle général du chauffage** : un seul contrôleur, les mêmes poids pour toutes les maisons, sans modèle ni réglage pour chacune. On l'essaie sur 1 000 maisons à Montréal en janvier, en montant une échelle en cinq marches.

> **Ce labo n'est pas un labo de RL classique.** On part du RL des manuels (une maison, un agent) et on monte jusqu'aux questions ouvertes de la recherche : généraliser à des bâtiments jamais vus, s'adapter *in context* sans changer ses poids, continuer d'apprendre quand le bâtiment change (meta-RL). C'est un aperçu de là où en est la recherche en RL et en contrôle général du CVC (*general HVAC control*) : les marches du haut marchent dans notre petit simulateur, mais restent un problème ouvert sur de vrais bâtiments.

> ⚠️ **Lancez d'abord la cellule d'installation.** Elle télécharge seule les fichiers compagnons (`lab_utils.py`, `rl_sb3.py`, `gru_ppo.py`, `schemas.py`, `images/`, `agents/`, la météo) et installe Stable-Baselines3 si besoin. Si elle n'atteint pas GitHub, elle affiche un bouton pour téléverser **le zip du lab**.

> Les corrections des cinq exercices sont déjà écrites. La météo est **synthétique**. Les agents sans mémoire viennent de Stable-Baselines3, ceux à mémoire de notre petit PPO à GRU ; tous sont précalculés sur CPU (600 itérations de PPO pour les agents SB3, 1 200 pour les agents à mémoire, une graine).

## 0 · Installation

In [ ]:
# Réglages de la séance
SHOW_WIDGETS = True      # curseurs interactifs (sinon, figures statiques)
RUN_TRAINING = False     # True : réentraîne l'agent « classique » en direct (~1 min 30 sur CPU)
METEO_REELLE = False     # True : remplace la météo synthétique par les janviers 2015-2024 d'Open-Meteo (Internet requis)

# Fichiers compagnons : lab_utils.py, rl_sb3.py, gru_ppo.py, schemas.py, images/, agents/, meteo_montreal.csv, tous dans lab_rl_prototype.zip
import os, sys, subprocess, urllib.request, zipfile
try:
    import stable_baselines3                         # Stable-Baselines3 (PPO), absent de Colab par défaut
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stable-baselines3"], check=True)
if not os.path.exists("lab_utils.py"):
    paquet = "lab_rl_prototype.zip"
    if not os.path.exists(paquet):
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/paabL/rl-hvac-lab/main/fr/lab_rl_prototype.zip", paquet)
            print("Fichiers compagnons téléchargés depuis GitHub.")
        except OSError:                             # GitHub inaccessible : on demande le zip
            try:
                from google.colab import files      # n'existe que dans Colab
            except ImportError:
                raise FileNotFoundError("lab_utils.py introuvable : ouvrez ce notebook depuis le dossier décompressé.")
            print("Téléchargement impossible. Cliquez sur « Choisir des fichiers » et choisissez lab_rl_prototype.zip.")
            zips = [nom for nom in files.upload() if nom.endswith(".zip")]
            assert zips, "Ce n'était pas un .zip : téléversez lab_rl_prototype.zip."
            paquet = zips[0]
    zipfile.ZipFile(paquet).extractall(".")
sys.path.insert(0, ".")

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from lab_utils import *
from rl_sb3 import *
from gru_ppo import AgentGRU, entrainer_gru
from schemas import schema_maison, schema_boucle, schema_agent, schema_prechauffage, schema_flotte
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
torch.set_num_threads(4)

if METEO_REELLE:
    telecharger_open_meteo().to_csv("meteo_montreal.csv", index=False)
meteo = charger_meteo()
print(f"{len(meteo)} heures de météo de janvier, de {meteo.t_ext.min():.0f} à {meteo.t_ext.max():.0f} °C")

### Ce qui est déjà codé

On ne code que l'essentiel, dans cinq exercices : la physique, REINFORCE, le tirage des maisons, le gradient du col et la demi-vie. Le reste est fourni, et l'on s'appuie sur des bibliothèques standard (Gymnasium, Stable-Baselines3) dès que le détail du code n'apporte rien. Seul le PPO à mémoire est maison : la tête de prédiction (*prediction head*) n'existe pas dans les bibliothèques.

| Fichier | Fonction ou classe | Ce qu'elle fait | Renvoie |
| --- | --- | --- | --- |
| `lab_utils.py` | `charger_meteo()` | les janviers de Montréal, au pas horaire | un DataFrame |
| | `tirer_maisons(n, largeur)` | tire n maisons au hasard dans la boîte des paramètres (exercice 3) | dict `UA`, `Cm`, `mu` |
| | `step(Ta, Tm, u, t_ext, xi)` | une heure du modèle de la maison (exercice 1) | `Ta`, `Tm`, kWh |
| | `Maisons(meteo, n)` | N maisons simulées d'un coup : `reset()`, `pas(actions)` | observations, rewards, coûts |
| | `thermostat(env)`, `thermostat_malin(env)` | les deux contrôleurs de référence, sans apprentissage | des actions |
| | `reinforce(meteo, maj=...)` | REINFORCE sur une maison, avec votre mise à jour (exercice 2) | poids, courbe |
| | `J(b, c)`, `grad_J(b, c)` | le jouet du col (exercice 4) | coût, gradient |
| `rl_sb3.py` | `MaisonGym(meteo)` | une maison à la convention Gymnasium (ci-dessous) | un `gymnasium.Env` |
| | `MaisonsVec(meteo, n)` | 256 maisons d'un coup, au format `VecEnv` de Stable-Baselines3 | pour entraîner vite |
| | `creer_agent(meteo)` | un `PPO` de SB3 sans mémoire (un MLP) | un agent SB3 |
| | `charger_agent(nom)` | un agent précalculé du dossier `agents/` | agent, historique |
| | `evaluer(politique, meteo, xi, debuts)` | joue une semaine sans exploration, sur des maisons fixées | `cout`, `inconfort`, `Ta`, mémoire `m` |
| | `banc_de_test(meteo, n)`, `cout_par_jour(d)` | n maisons et semaines de test, les mêmes pour tous ; le coût moyen par jour | |
| | `effet_effacement`, `effet_demenagement`, `sonde` | les trois expériences sur la mémoire (section 5) | surcoût en %, $R^2$ |
| `gru_ppo.py` | `AgentGRU(tete_prediction)` | notre agent à mémoire : un GRU, un actor, un critic, une tête de prédiction | un réseau PyTorch |
| | `entrainer_gru(agent, maisons, iterations)` | notre PPO à mémoire, en version courte (à appeler, pas à lire) | courbe d'apprentissage |
| `schemas.py` | `schema_...` | les schémas du notebook | une figure |

### L'environnement : la convention Gym

En RL, un **environnement** suit presque toujours la convention de [Gymnasium](https://gymnasium.farama.org) (l'ancien Gym d'OpenAI) :

- `observation_space` et `action_space` décrivent ce que l'agent voit et ce qu'il peut faire. Ici : 9 nombres (températures intérieure et extérieure, prix, pointe à venir, heure, dernière action) et `Discrete(5)`, cinq niveaux de puissance.
- `reset()` commence un épisode et renvoie `(observation, info)`.
- `step(action)` avance d'une heure et renvoie `(observation, reward, terminated, truncated, info)`. `terminated` signale une fin naturelle (ici, jamais) ; `truncated`, une fin par limite de temps (ici, après 168 heures).

`MaisonGym` est une maison à cette convention : n'importe quelle bibliothèque de RL peut l'utiliser. Pour entraîner, `MaisonsVec` fait avancer 256 maisons d'un coup, au format `VecEnv` de Stable-Baselines3. Voici la boucle agent-environnement, avec un agent qui joue au hasard :

In [ ]:
env = MaisonGym(meteo)                         # une maison tirée au hasard
obs, info = env.reset(seed=0)
total = 0.0
while True:
    action = env.action_space.sample()         # un agent qui joue au hasard
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    if terminated or truncated:
        break
print(env.observation_space, env.action_space)
print(f"Une semaine au hasard : {-total:.0f} $ de coût")

## 1 · Pourquoi contrôler le chauffage ?

- **L'enjeu.** Les bâtiments en service ont émis 9,9 Gt de CO₂ en 2024, soit 26 % du CO₂ lié à l'énergie (PNUE, *Global Status Report* 2026).
- **Le levier.** Le chauffage, la ventilation et la climatisation (CVC, *HVAC* en anglais) représentent 38 % de l'énergie des bâtiments (González-Torres et al., 2022).
- **La marge.** Une meilleure commande économise 13 à 28 % selon les revues de la littérature (Drgoňa et al., 2020).
- **La pointe.** Au Québec, les matins et soirs d'hiver, tout le monde chauffe en même temps. Hydro-Québec propose des tarifs de pointe pour étaler la demande : chauffer *avant* la pointe plutôt que pendant.

Préchauffer, c'est stocker de la chaleur dans les murs avant la pointe. Mais le bon plan dépend de la maison :

In [ ]:
schema_prechauffage("fr"); plt.show()

La commande prédictive (MPC) fait très bien ce travail... à condition d'avoir un modèle de chaque bâtiment. Construire et calibrer ce modèle maison par maison coûte trop cher pour des millions de logements. D'où notre fil rouge : **un contrôleur général**. Le RL peut-il apprendre un seul contrôleur qui s'adapte seul à chaque maison ?

In [ ]:
schema_flotte("fr"); plt.show()

### Les cinq régimes de l'échelle

On ne change jamais l'agent, seulement le monde dans lequel il s'entraîne. À chaque marche, il doit devenir un peu plus malin.

| Régime | En une phrase | Monde d'entraînement | Section |
| --- | --- | --- | --- |
| RL classique | apprendre à piloter **une** maison donnée | une seule maison | 3 |
| Robuste | trouver **un compromis**, le même pour toutes les maisons | des maisons tirées au hasard, sans mémoire | 4 |
| Mémoriser ou reconnaître | reconnaître une maison **déjà vue** à l'entraînement | quelques maisons, avec mémoire | 6 |
| In-context learning | estimer une maison **jamais vue** en l'observant, sans changer ses poids | une maison nouvelle à chaque épisode | 6 |
| Meta-RL | **continuer d'apprendre** quand la maison change en cours de route | des maisons qui changent pendant l'épisode | 7 |

**Le meta-RL en mots simples : apprendre à apprendre.** Pendant l'entraînement, l'agent voit tant de maisons, et des maisons qui changent, qu'il apprend une *façon* de s'adapter. Une fois déployé, ses poids ne bougent plus : c'est sa mémoire qui apprend la nouvelle maison, heure après heure, et qui la réapprend si elle change.

## 2 · Le problème : chauffer sans payer la pointe

> **Rappel : le vocabulaire du RL** (nouveau dans ce cours)
> - **État** (*state*) $s_t$ : ce que l'agent observe à l'heure $t$ (températures intérieure et extérieure, prix, heure).
> - **Action** $a_t$ : la puissance de la pompe à chaleur, de 0 à 100 % par paliers de 25 %.
> - **Reward** $r_t = -\text{coût}_t$, avec $\text{coût}_t = \text{prix}_t \times \text{kWh}_t + \lambda \times \text{inconfort}_t$ (K·h hors de 20-23 °C, $\lambda$ = 1 \$ par K·h).
> - **Retour** (*return*) $G_t = \sum_{k \ge t} r_k$ : tout ce qui reste à gagner jusqu'à la fin de la semaine. L'agent maximise le retour, pas la prochaine reward : payer un peu maintenant (préchauffer) peut économiser plus tard.
> - **Politique** (*policy*) $\pi(a \mid s)$ : la règle de décision, ce qu'on apprend.
> - **Épisode** : une semaine, 168 décisions horaires, dans une maison.

In [ ]:
schema_boucle("fr"); plt.show()

Une maison, c'est deux températures : l'air $T_a$, qui réagit en quelques minutes, et la masse $T_m$ (murs, planchers), qui réagit en quelques heures et reste **invisible** pour le thermostat. Chacune suit un bilan de chaleur, (capacité) × (vitesse de variation) = somme des flux :

$$C_a \frac{dT_a}{dt} = UA\,(T_{ext} - T_a) + H_{am}\,(T_m - T_a) + Q_{pompe} + Q_{int}, \qquad C_m \frac{dT_m}{dt} = H_{am}\,(T_a - T_m)$$

Trois paramètres $\xi$ changent d'une maison à l'autre et restent cachés : l'isolation $UA$, l'inertie $C_m$ et la taille de la pompe $\mu$. Le tarif est en pointe les jours de semaine de 6 h à 9 h et de 16 h à 20 h (0,50 \$/kWh contre 0,07).

In [ ]:
schema_maison("fr"); plt.show()

In [ ]:
# EXERCICE 1/5 · Un pas d'une heure du modèle à deux températures (Euler, 6 sous-pas de 10 min)
def mon_step(Ta, Tm, u, t_ext, xi, sous_pas=6):
    q_pompe = u * xi["mu"] * xi["UA"] * (T_CONSIGNE - T_DESIGN)     # chaleur fournie (W)
    dt = 3600.0 / sous_pas                                          # secondes
    for _ in range(sous_pas):
        # VOTRE CODE ICI : bilan de l'air (pertes, échange avec la masse, pompe, gains) puis de la masse
        dTa = (xi["UA"] * (t_ext - Ta) + HAM * (Tm - Ta) + q_pompe + QINT) / CA
        dTm = HAM * (Ta - Tm) / xi["Cm"]
        Ta, Tm = Ta + dt * dTa, Tm + dt * dTm
    return Ta, Tm, q_pompe / cop(t_ext) / 1000.0                    # électricité consommée (kWh) = chaleur / COP

xi_essai = tirer_maisons(5, rng=np.random.default_rng(0))
print("Test :", np.allclose(mon_step(20.0, 19.0, 0.5, -10.0, xi_essai), step(20.0, 19.0, 0.5, -10.0, xi_essai)))

Voici 64 maisons au hasard pendant une semaine, chauffées par le même thermostat proportionnel réglé à 21 °C. Les bandes grises sont les heures de pointe.

In [ ]:
xi_test, debuts_test = banc_de_test(meteo, n=256)       # 256 maisons et semaines de test, les mêmes pour tous les agents
d_th = evaluer(thermostat, meteo, xi_test, debuts_test)
pointe = est_pointe(np.arange(H))                       # 1 aux heures de pointe, 0 sinon
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(d_th["Ta"][:64], aspect="auto", cmap="coolwarm", vmin=17, vmax=26)
for t in range(H):
    if pointe[t] == 1:
        ax.axvspan(t - 0.5, t + 0.5, color="k", alpha=0.12, lw=0)
ax.set(xlabel="heure de la semaine", ylabel="maison", title="Le même thermostat dans 64 maisons : stable dans certaines, de fortes oscillations dans d'autres")
plt.colorbar(im, label="température intérieure (°C)"); plt.show()
print(f"Coût moyen : {cout_par_jour(d_th).mean():.2f} $/jour ; de {cout_par_jour(d_th).min():.2f} à {cout_par_jour(d_th).max():.2f} selon la maison")

## 3 · RL classique : une seule maison

> **Rappel : cours 2.** La régression logistique s'entraîne par $w_j \leftarrow w_j + \eta \sum_i (y_i - \hat y_i)\, x^i_j$.

> **Le policy gradient.** On cherche les poids $w$ qui maximisent le retour moyen $J(w) = \mathbb{E}_{\pi_w}[G_0]$. Le théorème du policy gradient donne
> $$\nabla_w J = \mathbb{E}\Big[\sum_t G_t\, \nabla_w \log \pi_w(a_t \mid s_t)\Big].$$
> Notre politique est une régression logistique sur 7 caractéristiques $x_t$ : $\pi_w(\text{chauffer} \mid s_t) = \sigma(w^\top x_t)$. Alors $\nabla_w \log \pi_w(a_t \mid s_t) = \big(a_t - \sigma(w^\top x_t)\big)\, x_t$, d'où la mise à jour de **REINFORCE** :
> $$w_j \leftarrow w_j + \eta \sum_t G_t\, \big(a_t - \sigma(w^\top x_t)\big)\, x_{t,j}.$$
> C'est la régression logistique du cours 2 : l'étiquette est l'action jouée, et chaque exemple est pondéré par le retour qui l'a suivi. Une action suivie d'un bon retour devient plus probable.

> **La baseline.** On remplace $G_t$ par $G_t - b_t$, avec $b_t$ la moyenne des retours à la même heure. Le gradient ne change pas en moyenne (car $\mathbb{E}[\nabla \log \pi] = 0$), mais il devient beaucoup moins bruité.

In [ ]:
# EXERCICE 2/5 · La mise à jour de REINFORCE
def ma_maj(w, X, A, P, G, eta):
    # X : (heures, maisons, 7) caractéristiques ; A : actions 0/1 ; P : probabilités π(chauffer) ; G : retours moins la baseline
    # VOTRE CODE ICI
    X2 = X.reshape(-1, X.shape[-1])             # un exemple par ligne
    poids = (G * (A - P)).reshape(-1)           # retour × (action - probabilité)
    gradient = X2.T @ poids / len(poids)        # comme la régression logistique : Xᵀ (y - ŷ) / n
    return w + eta * gradient

w, hist = reinforce(meteo, iterations=100, eta=5.0, maj=ma_maj, verbose=False)
noms = ["biais", "Ta - 21", "T_ext / 10", "pointe maintenant", "pointe bientôt", "sin heure", "cos heure"]
print(pd.Series(w, index=noms).round(2).to_string())
plt.plot(hist); plt.yscale("log"); plt.xlabel("itération"); plt.ylabel("coût ($/jour)"); plt.title("REINFORCE apprend à chauffer en quelques secondes"); plt.show()

**Question :** lisez les poids comme ceux d'une régression logistique. Pourquoi le poids de « Ta - 21 » est-il fortement négatif ? Et celui de « pointe maintenant » ? Pourquoi « pointe bientôt » reste-t-il proche de 0 ?

### Le policy gradient n'est qu'une famille parmi d'autres

| Famille | Ce qu'on apprend | Exemples | Dans ce lab |
| --- | --- | --- | --- |
| Policy gradient | la politique $\pi$ directement | REINFORCE, PPO | REINFORCE ici ; PPO pour tous les agents suivants |
| Actor-critic | $\pi$ et une valeur $V$ qui sert de baseline | A2C, PPO, SAC | nos agents PPO : un actor, un critic |
| Value-based | la valeur $Q(s, a)$ de chaque action, puis on prend la meilleure | Q-learning, DQN | — |
| Model-based | un modèle de la dynamique, puis on planifie | MPC, Dyna, MuZero | le MPC, dont l'oracle tient le rôle |
| Imitation | copier un expert | behavior cloning, DAgger | — |

> **De REINFORCE à PPO : le policy gradient avec advantages.** Trois retouches à la formule de REINFORCE, pour les poids $\theta$ d'un réseau.
>
> **1. L'advantage à la place du retour.** Un critic $V(s)$ apprend à prédire le retour. L'**advantage** $A_t = Q(s_t, a_t) - V(s_t)$ dit si l'action a fait mieux ou moins bien que prévu ; on l'estime par $\hat A_t = G_t - V(s_t)$. C'est la baseline, mais apprise :
> $$\nabla_\theta J = \mathbb{E}\Big[\sum_t \hat A_t\, \nabla_\theta \log \pi_\theta(a_t \mid s_t)\Big].$$
>
> **2. Une estimation moins bruitée (GAE).** Au lieu d'attendre la fin de la semaine, on compare chaque heure à ce que prévoyait le critic : $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$, puis $\hat A_t = \delta_t + \gamma\lambda\,\delta_{t+1} + (\gamma\lambda)^2\,\delta_{t+2} + \dots$ (ici $\gamma = 0{,}97$ et $\lambda = 0{,}95$).
>
> **3. Le garde-fou (*clipping*).** PPO réutilise chaque batch d'expérience plusieurs fois. Pour ne pas trop s'éloigner de la politique qui a joué, il maximise
> $$L^{\text{clip}}(\theta) = \mathbb{E}_t\Big[\min\big(\rho_t\,\hat A_t,\ \operatorname{clip}(\rho_t,\, 1-\varepsilon,\, 1+\varepsilon)\,\hat A_t\big)\Big], \qquad \rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{anc}}(a_t \mid s_t)}, \quad \varepsilon = 0{,}2.$$
> Au début de chaque mise à jour, $\rho_t = 1$ et ce gradient est exactement celui du point 1. Dès que le ratio $\rho_t$ sort de $[0{,}8 ; 1{,}2]$ dans le sens qui améliore, le gradient s'annule : une mise à jour ne peut pas trop changer la politique.
>
> La perte totale ajoute l'erreur du critic, $(V(s_t) - G_t)^2$, et un petit bonus d'entropie pour continuer d'explorer. On ne recode pas PPO : pour les agents sans mémoire, on prend celui de **Stable-Baselines3** (SB3). Pour les agents à mémoire, `gru_ppo.py` contient une version courte du même algorithme, avec la tête de prédiction que les bibliothèques n'offrent pas.

Un agent tout-ou-rien reste grossier. Avec cinq niveaux de puissance, une mémoire (un GRU) et PPO, on obtient l'agent « classique ». Entraîné sur cette seule maison, il apprend à préchauffer avant la pointe. L'entraîner tient en deux lignes, et environ 1 min 30 sur CPU :

In [ ]:
if RUN_TRAINING:
    agent_classique = AgentGRU()                                                 # notre agent à mémoire (gru_ppo.py)
    entrainer_gru(agent_classique, Maisons(meteo, 256, xi=maison_nominale(256)), iterations=60)   # 256 copies de la maison
else:
    agent_classique, _ = charger_agent("classique")

xi_nom = maison_nominale(256)
for nom, pol in [("thermostat", thermostat), ("thermostat qui préchauffe", thermostat_malin), ("agent classique", agent_classique)]:
    print(f"{nom:28s} {cout_par_jour(evaluer(pol, meteo, xi_nom, debuts_test)).mean():6.2f} $/jour")

fig, ax = plt.subplots(figsize=(11, 3.5))
for nom, pol in [("thermostat", thermostat), ("agent classique", agent_classique)]:
    ax.plot(evaluer(pol, meteo, xi_nom, debuts_test)["Ta"][0, :72], label=nom)
for t in range(72):
    if pointe[t] == 1:
        ax.axvspan(t - 0.5, t + 0.5, color="k", alpha=0.1, lw=0)
ax.axhspan(*BANDE, color="g", alpha=.08); ax.set(xlabel="heure", ylabel="°C", title="L'agent monte avant la pointe et redescend pendant"); ax.legend(); plt.show()

> **Sur une maison connue, le RL ne sert à rien : un MPC suffit.** Si l'on connaît le modèle de la maison (ses paramètres $\xi$), la commande prédictive optimise directement les prochaines heures à partir des prévisions de météo et de prix, sans aucun entraînement. Dans nos chiffres, l'**oracle**, un agent à qui l'on donne $\xi$, tient ce rôle. Le RL devient intéressant quand on veut **un seul contrôleur pour toutes les maisons**, sans modèle de chacune.

> **Vers le contrôle général.** L'agent classique ne connaît qu'une maison : il n'est pas général du tout. Voyons ce qu'il fait ailleurs.

## 4 · Robuste : un compromis pour toutes les maisons

Pour piloter mille maisons, on entraîne sur des maisons tirées au hasard : c'est **l'augmentation de données** des cours 1 et 7, appelée en RL *domain randomization*. Un agent sans mémoire (un MLP qui ne voit que l'observation du moment, entraîné par le PPO de SB3) ne peut pas distinguer les maisons : au mieux, il apprend un compromis qui marche partout à peu près.

In [ ]:
# EXERCICE 3/5 · Tirer n maisons : ξ = nominal × exp(U(-w, w)), w = largeur × log(sqrt(facteur))
def mes_maisons(n, largeur=1.0, rng=np.random.default_rng(0)):
    # VOTRE CODE ICI
    maisons = {}
    for k in NOMINAL:
        w = largeur * np.log(np.sqrt(FACTEUR[k]))
        maisons[k] = NOMINAL[k] * np.exp(rng.uniform(-w, w, n))
    return maisons

xi_demo = mes_maisons(2000)
for k in xi_demo:
    print(k, "de", round(xi_demo[k].min(), 2), "à", round(xi_demo[k].max(), 2))

In [ ]:
oracle, _ = charger_agent("oracle")      # reçoit ξ et la température de la masse : l'étalon, pas un vrai contrôleur
robuste, _ = charger_agent("robuste")    # PPO sans mémoire, entraîné sur toutes les maisons
res = {}
res["thermostat"] = cout_par_jour(d_th)
res["classique (une maison)"] = cout_par_jour(evaluer(agent_classique, meteo, xi_test, debuts_test))
res["robuste (sans mémoire)"] = cout_par_jour(evaluer(robuste, meteo, xi_test, debuts_test))
res["oracle (voit ξ)"] = cout_par_jour(evaluer(oracle, meteo, xi_test, debuts_test, privilegie=True))
for k, v in res.items():
    print(f"{k:26s} {v.mean():6.2f} $/jour   écart à l'oracle {100 * (v.mean() / res['oracle (voit ξ)'].mean() - 1):+6.1f} %")

ecart = 100 * (res["robuste (sans mémoire)"] / res["oracle (voit ξ)"] - 1)
plt.figure(figsize=(6, 4.5))
sc = plt.scatter(xi_test["UA"], xi_test["mu"], c=ecart, cmap="magma_r", s=25)
plt.colorbar(sc, label="surcoût du robuste face à l'oracle (%)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("isolation UA (W/K, plus = passoire)"); plt.ylabel("taille de la pompe μ")
plt.title(f"Le prix de l'ignorance : le même compromis\ncoûte jusqu'à +{ecart.max():.0f} % selon la maison"); plt.show()

> **Vers le contrôle général.** Un seul contrôleur pour toutes les maisons, c'est fait : environ +11 % au-dessus de l'oracle. Mais il est aveugle à la maison. Pour faire mieux, il faudrait la reconnaître.

## 5 · L'agent écoute-t-il sa mémoire ?

> **Pourquoi une mémoire peut reconnaître la maison.** Chauffer, puis regarder comment la température répond, c'est ce que font les ingénieurs pour modéliser un bâtiment : la **system identification**. Une maison passoire refroidit vite quand on coupe ; une petite pompe réchauffe lentement quand on pousse. Avec quelques heures d'historique (puissance demandée, températures), une simple régression estimerait $UA$, $C_m$ et $\mu$. Une mémoire peut faire ce calcul implicitement, sans qu'on lui dise quoi chercher... si l'entraînement l'y pousse.

On donne donc une mémoire à l'agent : un **GRU**, que l'on peut lire comme un MLP qui relit sa propre sortie à chaque heure, $h_t = f(W[x_t, h_{t-1}] + w_0)$. Au-dessus, trois têtes (*heads*).

In [ ]:
schema_agent("fr"); plt.show()

> **Actor-critic.** L'actor choisit l'action. Le critic $V(h_t)$ prédit le retour attendu, et sert de baseline : l'actor apprend de l'**advantage** $A_t = G_t - V(h_t)$, « mieux ou moins bien que prévu ». C'est l'advantage de la formule de PPO (section 3) : ici, le critic lit la mémoire $h_t$.

> **Le self-predictive RL.** La reward dit si l'action était bonne, pas ce qu'est la maison : c'est un signal faible et bruité pour apprendre à l'identifier. La troisième tête ajoute une tâche auto-supervisée : prédire la température de l'heure suivante à partir de la mémoire et de l'action. La cible est gratuite, c'est le thermomètre. Pour bien prédire, la mémoire doit contenir la maison. Faire prédire à l'agent sa propre suite aide beaucoup quand l'état est caché (Schwarzer et al., 2021 (UdeM, Mila) ; Ni et al., 2024 (UdeM, Mila)). Ici, sa perte s'ajoute simplement à celle de PPO, dans `gru_ppo.py`.

Un jouet à deux poids montre pourquoi le RL seul a du mal. La mémoire écrit $m = c\,\xi + \varepsilon$ (ce qu'elle retient de la maison, plus du bruit), l'action la lit $a = b\,m$, et l'action idéale est $\xi$ :

$$J(b, c) = \sigma_\xi^2\,(bc - 1)^2 + b^2 \sigma_\varepsilon^2$$

En $(0, 0)$, la politique robuste, lire ne rapporte rien tant que la mémoire est vide, et écrire ne rapporte rien tant que personne ne lit : le gradient est nul. C'est un **col** (un point selle, *saddle point*), et la descente de gradient peut y traîner longtemps.

In [ ]:
# EXERCICE 4/5 · Le gradient du jouet
def mon_grad_J(b, c, s_xi=1.0, s_eps=1.0):
    # VOTRE CODE ICI : dJ/db, dJ/dc
    erreur = b * c - 1
    dJ_db = 2 * s_xi**2 * erreur * c + 2 * b * s_eps**2
    dJ_dc = 2 * s_xi**2 * erreur * b
    return dJ_db, dJ_dc

print("Gradient au col (0, 0) :", mon_grad_J(0.0, 0.0), "   Test :", np.allclose(mon_grad_J(.3, -.7), grad_J(.3, -.7)))

def paysage(s_xi=1.0, tete=0.0):
    grille = np.linspace(-1.5, 1.5, 200)
    B, Cc = np.meshgrid(grille, grille)
    cout_affiche = J(B, Cc, s_xi) + tete * (1 - Cc) ** 2
    plt.figure(figsize=(5.5, 5))
    plt.contourf(B, Cc, np.log(cout_affiche + 1e-3), 30, cmap="Blues_r")
    rng = np.random.default_rng(1)
    for _ in range(20):
        b, c = rng.normal(0, 1e-3, 2)        # on part tout près du col
        traj = [(b, c)]
        for _ in range(150):
            gb, gc = mon_grad_J(b, c, s_xi)
            gc += -2 * tete * (1 - c)        # la tête de prédiction : un gradient pour écrire, sans passer par la lecture
            b, c = b - 0.03 * gb, c - 0.03 * gc
            traj.append((b, c))
        traj = np.array(traj)
        plt.plot(traj[:, 0], traj[:, 1], "w-", lw=1)
        plt.plot(traj[-1, 0], traj[-1, 1], "wo", ms=4)
    b_cible = s_xi**2 / (s_xi**2 + 1.0)       # mémoire écrite (c = 1), lue au mieux
    plt.plot(0, 0, "r*", ms=15, label="col : politique robuste (mémoire ni écrite ni lue)")
    plt.plot(b_cible, 1.0, "P", color="#2ecc40", mec="k", ms=15, label="objectif : la mémoire contient la maison, l'action la lit")
    plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5)
    plt.xlabel("b : lire la mémoire"); plt.ylabel("c : écrire la maison dans la mémoire")
    plt.title(f"σξ = {s_xi:.1f}, tête de prédiction = {tete:.1f}")
    plt.legend(loc="lower left", fontsize=8.5)
    plt.show()

if SHOW_WIDGETS:
    import ipywidgets as W
    W.interact(paysage, s_xi=W.FloatSlider(1.0, min=0.3, max=2.0, step=0.1, description="largeur σξ"),
               tete=W.FloatSlider(0.0, min=0.0, max=1.0, step=0.1, description="tête"))
else:
    paysage(1.0, 0.0); paysage(1.0, 0.5)

La **croix verte** est là où l'on veut aller : la politique adaptative, où la mémoire contient la maison ($c \neq 0$) et où l'action s'en sert ($b \neq 0$). Son coût est plus bas que celui du col, parce que l'action s'ajuste à chaque maison au lieu d'un compromis. L'**étoile rouge** est là où l'entraînement sans tête reste coincé : lire une mémoire vide ne sert à rien, et écrire une mémoire que personne ne lit non plus.

**Question :** élargir la boîte des maisons (σξ) creuse la vallée. Fait-il quitter le col plus vite ? Qu'est-ce qui change quand on allume la tête de prédiction ?

Même expérience sur les vrais agents : un GRU entraîné par PPO **sans** tête de prédiction, et le même **avec** la tête.

In [ ]:
col, _ = charger_agent("col")
contexte, _ = charger_agent("contexte")
lignes = []
for nom, ag in [("GRU sans tête (col)", col), ("GRU avec tête", contexte)]:
    d = evaluer(ag, meteo, xi_test, debuts_test)
    res[nom] = cout_par_jour(d)
    ligne = {"agent": nom, "cout": res[nom].mean()}
    ligne["effacer_memoire"] = effet_effacement(ag, meteo, xi_test, debuts_test)
    ligne["demenager"] = effet_demenagement(ag, meteo, xi_test, debuts_test)
    r2 = sonde(d, (6, 24, 72, 167))
    for t in r2:
        ligne[f"R2 sonde h{t}"] = r2[t]
    lignes.append(ligne)
tab = pd.DataFrame(lignes).set_index("agent").round(2)
tab

- **cout** : \$/jour sur les 256 maisons de test.
- **effacer_memoire** : on vide la mémoire à l'heure 72 ; surcoût sur les 24 heures suivantes (%).
- **demenager** : on échange les mémoires de deux maisons à l'heure 72, chaque agent « croit » être dans l'autre maison ; surcoût sur les 24 heures suivantes (%).
- **R2 sonde** : une sonde (*linear probe*), ici une Ridge de scikit-learn, lit $\xi$ dans la mémoire ; $R^2$ sur des maisons jamais vues par la sonde. Elle mesure ce que la mémoire *sait*, pas si la politique *s'en sert*.

**Question :** quel agent *écoute* vraiment sa mémoire ? Lequel sait quelque chose sans s'en servir ?

> **Vers le contrôle général.** Avec la tête, la mémoire lit la maison dans l'historique et la politique s'en sert : environ +3 % au-dessus de l'oracle, contre +6 % sans tête et +11 % sans mémoire.

## 6 · Mémoriser ou reconnaître, puis in-context learning

> **Rappel : cours 3 et 4.** L'overfitting, c'est mémoriser l'ensemble d'entraînement au lieu de comprendre.

Même agent (GRU avec tête), trois mondes d'entraînement : 4 maisons, 16 maisons, ou une maison nouvelle à chaque épisode. On le teste sur ses maisons d'entraînement et sur des maisons qu'il n'a jamais vues.

In [ ]:
lignes = []
for nom, k in [("memoriser_4", 4), ("memoriser_16", 16), ("contexte", None)]:
    ag, _ = charger_agent(nom)
    if k:
        xi_k = tirer_maisons(k, 1.0, np.random.default_rng(11))     # les k maisons de son entraînement
        idx = np.arange(256) % k
        xi_train = {key: v[idx] for key, v in xi_k.items()}
    else:
        xi_train = tirer_maisons(256, 1.0, np.random.default_rng(99))
    ref_train = cout_par_jour(evaluer(oracle, meteo, xi_train, debuts_test, privilegie=True)).mean()
    c_train = cout_par_jour(evaluer(ag, meteo, xi_train, debuts_test)).mean()
    if k:
        c_test = cout_par_jour(evaluer(ag, meteo, xi_test, debuts_test)).mean()
    else:
        c_test = res["GRU avec tête"].mean()
    if k:
        monde = f"{k} maisons"
    else:
        monde = "toujours nouvelles"
    lignes.append({"monde": monde,
                   "maisons d'entraînement": 100 * (c_train / ref_train - 1),
                   "maisons nouvelles": 100 * (c_test / res["oracle (voit ξ)"].mean() - 1)})
gen = pd.DataFrame(lignes).set_index("monde").round(1)
gen.plot.bar(rot=0, figsize=(8, 3.5), ylabel="écart à l'oracle (%)", title="Peu de maisons : il mémorise. Beaucoup : il généralise")
plt.show(); gen

**In-context learning.** Les poids sont maintenant figés : seule la mémoire change pendant que l'agent observe la maison. Chaque point est une maison de test, placée par les deux premières composantes principales de sa mémoire (64 nombres) et colorée par son isolation.

In [ ]:
from sklearn.decomposition import PCA
d = evaluer(contexte, meteo, xi_test, debuts_test)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, t in zip(axs, (2, 96)):
    p = PCA(2).fit_transform(d["m"][:, t])
    s = ax.scatter(p[:, 0], p[:, 1], c=np.log(xi_test["UA"]), cmap="viridis", s=15)
    ax.set(title=f"Mémoire à l'heure {t}", xlabel="CP1", ylabel="CP2")
plt.colorbar(s, ax=axs, label="log isolation UA"); fig.suptitle("Lire dans la mémoire : les maisons se rangent par isolation au fil des heures", y=1.04); plt.show()

> **Vers le contrôle général.** L'agent identifie en quelques heures une maison qu'il n'a jamais vue, sans toucher à ses poids, comme un LLM qui saisit une tâche à partir des exemples de son prompt. Reste un cas : la maison qui change.

## 7 · Meta-RL : quand la maison change en cours de route

Jusqu'ici, une maison reste la même toute la semaine. Les vraies maisons changent : une pompe faiblit, une fenêtre reste ouverte. Un agent qui a identifié la maison une fois doit continuer de l'identifier.

Pourquoi c'est difficile : une estimation qui fait la moyenne de **tout** le passé (`expanding().mean()`) réagit de plus en plus lentement. Après un saut, elle met autant de temps à se corriger à moitié que l'âge qu'elle avait au moment du saut : c'est sa **demi-vie**. Une moyenne qui oublie (`ewm`, moyenne exponentielle) garde une demi-vie constante.

In [ ]:
# EXERCICE 5/5 · La demi-vie d'une estimation après un saut
def demi_vie(t0, methode, n=400):
    x = pd.Series(np.r_[np.zeros(t0), np.ones(n - t0)])        # le paramètre saute de 0 à 1 à l'heure t0
    # VOTRE CODE ICI : estimation cumulée ou exponentielle (demi-vie 12 h)
    if methode == "cumulée":
        estimation = x.expanding().mean()
    else:
        estimation = x.ewm(halflife=12).mean()
    for h in range(n - t0):                                   # première heure après le saut où l'estimation passe 0,5
        if estimation[t0 + h] >= 0.5:
            return h

for t0 in [12, 24, 48, 96, 192]:
    print(f"saut à l'heure {t0:3d} : demi-vie cumulée {demi_vie(t0, 'cumulée'):3d} h, exponentielle {demi_vie(t0, 'exponentielle'):3d} h")

Dans le simulateur, la dérive tombe à une heure au hasard : la pompe perd la moitié de sa puissance, ou l'isolation se dégrade d'un facteur 1,8. On compare l'agent entraîné sans dérive à celui entraîné avec.

In [ ]:
derive, _ = charger_agent("derive")
oracle_d, _ = charger_agent("oracle_derive")
env_d = Maisons(meteo, 256, derive=True, seed=123); env_d.reset(xi=xi_test, debuts=debuts_test)   # les mêmes sauts que dans evaluer()
t_saut = env_d.t_saut
def cout_apres_saut(d):
    """Coût moyen par jour dans les 48 heures qui suivent le saut de chaque maison."""
    total = 0.0
    for i in range(len(t_saut)):
        total += d["cout"][i, t_saut[i]:t_saut[i] + 48].sum()
    return total / len(t_saut) / 2

agents_derive = [("GRU avec tête, entraîné sans dérive", contexte, False),
                 ("GRU avec tête, entraîné avec dérive", derive, False),
                 ("oracle (voit ξ en direct)", oracle_d, True)]
semaine, apres, traces = {}, {}, {}
for nom, ag, priv in agents_derive:
    d = evaluer(ag, meteo, xi_test, debuts_test, privilegie=priv, derive=True)
    semaine[nom] = cout_par_jour(d).mean()
    apres[nom] = cout_apres_saut(d)
    traces[nom] = d["Ta"]
ref = apres["oracle (voit ξ en direct)"]
derive_tab = pd.DataFrame({"coût sur la semaine ($/jour)": semaine, "coût dans les 48 h après le saut ($/jour)": apres})
derive_tab["écart à l'oracle après le saut (%)"] = 100 * (derive_tab["coût dans les 48 h après le saut ($/jour)"] / ref - 1)
derive_tab.round(2)

In [ ]:
# une maison dont la pompe perd la moitié de sa puissance, là où l'écart entre les deux agents est le plus net
noms = list(traces)

def froid_apres_saut(Ta, i):
    """K·h sous 20 °C dans les 48 heures qui suivent le saut de la maison i."""
    return np.maximum(BANDE[0] - Ta[i, t_saut[i]:t_saut[i] + 48], 0).sum()

i, meilleur_ecart = None, -1.0
for j in range(len(t_saut)):
    pompe_cassee = env_d.type_saut[j] == 0
    oracle_au_chaud = froid_apres_saut(traces[noms[2]], j) < 2
    if pompe_cassee and oracle_au_chaud:
        ecart_j = froid_apres_saut(traces[noms[0]], j) - froid_apres_saut(traces[noms[1]], j)
        if ecart_j > meilleur_ecart:
            i, meilleur_ecart = j, ecart_j

def demi_vie_ecart(Ta, Ta_ref, t0, lissage=6):
    """Heures après le saut jusqu'à ce que l'écart à l'oracle retombe sous la moitié de son maximum."""
    g = pd.Series(Ta_ref - Ta).rolling(lissage, min_periods=1).mean().values[t0:]
    pic = int(g.argmax())
    sous = np.where(g[pic:] < g[pic] / 2)[0]
    return pic + int(sous[0]) if len(sous) else len(g), g[pic]

fig, ax = plt.subplots(figsize=(11, 4))
for nom, Ta in traces.items():
    ax.plot(Ta[i], label=nom, lw=1.2)
t0 = t_saut[i]
ax.axvline(t0, color="r", ls="--", label="la pompe perd 50 %"); ax.axhspan(*BANDE, color="g", alpha=.08)
bas = min(traces[nom][i].min() for nom in noms)
ax.set_ylim(bas - 1.3, None)
for k in [0, 1]:
    nom, y = noms[k], bas - 0.45 - 0.5 * k
    n12, pic = demi_vie_ecart(traces[nom][i], traces[noms[2]][i], t0)
    if pic > 0.2:
        ax.annotate("", xy=(t0 + n12, y), xytext=(t0, y), arrowprops=dict(arrowstyle="<->", color=f"C{k}", lw=1.5))
        ax.text(t0 + n12 + 2, y, f"demi-vie ≈ {n12} h", va="center", color=f"C{k}", fontsize=10)
    else:
        ax.text(t0 + 2, y, "écart à l'oracle négligeable", va="center", color=f"C{k}", fontsize=10)
ax.set(xlabel="heure", ylabel="°C", title="Après le saut : qui s'adapte, et en combien de temps ?"); ax.legend(fontsize=9, loc="upper right"); plt.show()

La **demi-vie** se lit sur la figure : le temps, après le saut, pour que l'écart de température avec l'oracle retombe à la moitié de son maximum.

**Question :** comparez avec l'exercice 5. Lequel des deux agents se comporte comme la moyenne cumulée, lequel comme la moyenne exponentielle ?

> **Vers le contrôle général.** Entraîné sur des maisons qui changent, l'agent continue d'apprendre après son déploiement, avec des poids figés. C'est le meta-RL : la marche du haut.

## 8 · Synthèse : où chaque agent s'est arrêté

In [ ]:
synthese = pd.DataFrame({"coût ($/jour)": {k: v.mean() for k, v in res.items()}})
synthese["écart à l'oracle (%)"] = 100 * (synthese["coût ($/jour)"] / res["oracle (voit ξ)"].mean() - 1)
synthese.sort_values("coût ($/jour)", ascending=False).round(2)

| Régime | Monde d'entraînement | Agent du jour | Ce qu'il sait faire |
| --- | --- | --- | --- |
| RL classique | une maison | agent classique | bat le thermostat chez lui, échoue ailleurs |
| Robuste | maisons tirées au hasard | PPO sans mémoire ; GRU sans tête | un compromis à +11 % de l'oracle ; +6 % pour le GRU sans tête, qui lit peu sa mémoire |
| Mémoriser ou reconnaître | 4 ou 16 maisons | GRU avec tête | avec 4 maisons : +3,5 % chez lui, +15 % ailleurs ; avec 16 : +2 % et +2 % |
| In-context learning | une maison nouvelle à chaque épisode | GRU avec tête | identifie une maison jamais vue, à +3 % de l'oracle |
| Meta-RL | la maison change pendant l'épisode | GRU avec tête, entraîné avec dérive | après une panne, +7 % de l'oracle, contre +13 % sans dérive à l'entraînement |

**Le contrôle général, en bref.** Un MPC a besoin du modèle de chaque maison ; notre dernier agent n'a besoin que de son historique. À grande échelle, le projet de recherche NOMAD, dont ce labo est une version réduite, est resté près du col : la mémoire savait un peu, mais la politique ne la lisait pas. Le faire marcher sur de vrais bâtiments reste une question ouverte.